In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Environment Ready")

Environment Ready


In [3]:
import pandas as pd
import sklearn
import xgboost

print("Pandas:", pd.__version__)
print("Scikit-Learn:", sklearn.__version__)
print("XGBoost:", xgboost.__version__)

Pandas: 3.0.1
Scikit-Learn: 1.8.0
XGBoost: 3.2.0


In [4]:
import os

folders = [
    "../data/raw",
    "../data/processed",
    "../models",
    "../reports",
]

for folder in folders:
    print(folder, "Exists:", os.path.exists(folder))

../data/raw Exists: True
../data/processed Exists: True
../models Exists: True
../reports Exists: True


# UK Energy Demand Forecasting

## Business Problem

Energy providers need accurate forecasts of future electricity demand to optimize generation capacity, reduce operational costs, and improve grid reliability.

## Objective

Predict next-day UK electricity demand using historical demand, weather conditions, and calendar-based features.

## Workflow

1. Data Collection
2. Data Cleaning
3. Exploratory Data Analysis
4. Feature Engineering
5. Model Training
6. Model Evaluation
7. SHAP Explainability
8. Power BI Dashboard
9. AWS SageMaker Deployment

In [5]:
df = pd.DataFrame({
    "Date": ["2025-01-01", "2025-01-02"],
    "Demand_MW": [32000, 32500]
})

df

,Date,Demand_MW
0,2025-01-01,32000
1,2025-01-02,32500


In [7]:
import requests
import pandas as pd
import time

BASE_URL = "https://api.neso.energy/api/3/action/datastore_search_sql"

resource_id = "8a4a771c-3929-4e56-93ad-cdf13219dea5"

all_data = []
offset = 0
batch_size = 10000

while True:

    sql_query = f"""
    SELECT *
    FROM "{resource_id}"
    WHERE "SETTLEMENT_DATE" >= '2022-01-01'
    AND "SETTLEMENT_DATE" <= '2026-06-03'
    ORDER BY "_id"
    LIMIT {batch_size}
    OFFSET {offset}
    """

    response = requests.get(
        BASE_URL,
        params={"sql": sql_query}
    )

    records = response.json()["result"]["records"]

    if len(records) == 0:
        break

    all_data.extend(records)

    print(f"Downloaded {len(all_data):,} rows")

    offset += batch_size

    time.sleep(0.5)

df = pd.DataFrame(all_data)

print("\nFinal Shape:")
print(df.shape)

Downloaded 6,382 rows

Final Shape:
(6382, 24)


In [8]:
df.to_csv(
    "../data/raw/energy/historic_demand_2026.csv",
    index=False
)

In [9]:
import pandas as pd
from pathlib import Path

energy_dir = Path("../data/raw/energy")

energy_dir.mkdir(parents=True, exist_ok=True)

print("Folder Ready")

Folder Ready


In [10]:
import requests
from pathlib import Path

energy_dir = Path("../data/raw/energy")

files = {
    "2001": "https://api.neso.energy/dataset/8f2fe0af-871c-488d-8bad-960426f24601/resource/e8608e9a-f56c-457f-b9e7-bfffcfd19731/download/demanddata_2001.csv",
    "2002": "https://api.neso.energy/dataset/8f2fe0af-871c-488d-8bad-960426f24601/resource/4daaac31-ae56-461e-8efe-6b33e147225a/download/demanddata_2002.csv",
    "2003": "https://api.neso.energy/dataset/8f2fe0af-871c-488d-8bad-960426f24601/resource/30650247-acbf-414d-8ae2-2aa4aac57537/download/demanddata_2003.csv",
    "2004": "https://api.neso.energy/dataset/8f2fe0af-871c-488d-8bad-960426f24601/resource/5f5c076c-d3a3-4f33-a137-501cc3d2ca9b/download/demanddata_2004.csv",
    "2005": "https://api.neso.energy/dataset/8f2fe0af-871c-488d-8bad-960426f24601/resource/42a41acd-53b6-450d-af1b-4a92a352895c/download/demanddata_2005.csv",
    "2006": "https://api.neso.energy/dataset/8f2fe0af-871c-488d-8bad-960426f24601/resource/949bb4a4-8374-4730-89ef-302d82428d2c/download/demanddata_2006.csv",
    "2007": "https://api.neso.energy/dataset/8f2fe0af-871c-488d-8bad-960426f24601/resource/c0da767c-447d-48d1-93fd-ca3955d078ae/download/demanddata_2007.csv",
    "2008": "https://api.neso.energy/dataset/8f2fe0af-871c-488d-8bad-960426f24601/resource/ec63adc8-e8a1-45cc-a593-5953b1ede7b1/download/demanddata_2008.csv",
    "2025": "https://api.neso.energy/dataset/8f2fe0af-871c-488d-8bad-960426f24601/resource/b2bde559-3455-4021-b179-dfe60c0337b0/download/demanddata_2025.csv",
    "2026": "https://api.neso.energy/dataset/8f2fe0af-871c-488d-8bad-960426f24601/resource/8a4a771c-3929-4e56-93ad-cdf13219dea5/download/demanddata_2026.csv"
}

for year, url in files.items():

    output_file = energy_dir / f"demanddata_{year}.csv"

    r = requests.get(url)

    with open(output_file, "wb") as f:
        f.write(r.content)

    print(f"Downloaded {year}")

Downloaded 2001
Downloaded 2002
Downloaded 2003
Downloaded 2004
Downloaded 2005
Downloaded 2006
Downloaded 2007
Downloaded 2008
Downloaded 2025
Downloaded 2026


In [11]:
import os

files = os.listdir("../data/raw/energy")

print(f"Total Files: {len(files)}")

files[:10]

Total Files: 11


['demanddata_2001.csv',
 'demanddata_2002.csv',
 'demanddata_2003.csv',
 'demanddata_2004.csv',
 'demanddata_2005.csv',
 'demanddata_2006.csv',
 'demanddata_2007.csv',
 'demanddata_2008.csv',
 'demanddata_2025.csv',
 'demanddata_2026.csv']

In [14]:
import requests

url = "https://api.neso.energy/api/3/action/datapackage_show?id=historic-demand-data"

response = requests.get(url)

data = response.json()

print("Success:", data["success"])

print("Number of resources:", len(data["result"]["resources"]))

Success: True
Number of resources: 27


In [15]:
import requests
import pandas as pd

url = "https://api.neso.energy/api/3/action/datapackage_show?id=historic-demand-data"

response = requests.get(url)

data = response.json()

resources = data["result"]["resources"]

print("Total Resources:", len(resources))

for r in resources:
    print(r["name"])

Total Resources: 27
frequently_asked_questions_(faq)
historic_demand_data_2009
historic_demand_data_2010
historic_demand_data_2011
historic_demand_data_2012
historic_demand_data_2013
historic_demand_data_2014
historic_demand_data_2015
historic_demand_data_2016
historic_demand_data_2017
historic_demand_data_2018
historic_demand_data_2019
historic_demand_data_2020
historic_demand_data_2021
historic_demand_data_2022
historic_demand_data_2023
historic_demand_data_2024
historic_demand_data_2025
historic_demand_data_2001
historic_demand_data_2002
historic_demand_data_2003
historic_demand_data_2004
historic_demand_data_2005
historic_demand_data_2006
historic_demand_data_2007
historic_demand_data_2008
historic_demand_data_2026


In [17]:
resources[1]

{'hash': '',
 'description': 'Historic electricity demand, interconnector, wind and solar outturn data for 2009.',
 'mediatype': 'text/csv',
 'package_id': '8f2fe0af-871c-488d-8bad-960426f24601',
 'url_type': 'upload',
 'position': 1,
 'descriptionHtml': '<p>Historic electricity demand, interconnector, wind and solar outturn data for 2009.</p>',
 'path': 'https://api.neso.energy/dataset/8f2fe0af-871c-488d-8bad-960426f24601/resource/ed8a37cb-65ac-4581-8dbc-a3130780da3a/download/demanddata_2009.csv',
 'datastore_active': True,
 'id': 'ed8a37cb-65ac-4581-8dbc-a3130780da3a',
 'name': 'historic_demand_data_2009',
 'metadata_modified': '2023-07-24T12:16:54.514200',
 'created': '2021-03-12T16:50:57.944407',
 'format': 'CSV',
 'token': '4c6b21f7a312f5524dc08daf97e6f34d9f39737a7b802b9b6c0311954f5ab34c',
 'state': 'active',
 'last_modified': '2023-07-24T12:16:55.273828',
 'title': 'Historic Demand Data 2009',
 'revision_id': '0c12f11f-413b-4962-8647-301b40fc5f90'}

In [18]:
r["path"]

'https://api.neso.energy/dataset/8f2fe0af-871c-488d-8bad-960426f24601/resource/243ca76a-eefe-419d-a4b5-397073deb100/download/faq-neso.docx'

In [20]:
for r in resources:

    if "historic_demand_data_" in r["name"]:
        print(r["name"])

historic_demand_data_2009
historic_demand_data_2010
historic_demand_data_2011
historic_demand_data_2012
historic_demand_data_2013
historic_demand_data_2014
historic_demand_data_2015
historic_demand_data_2016
historic_demand_data_2017
historic_demand_data_2018
historic_demand_data_2019
historic_demand_data_2020
historic_demand_data_2021
historic_demand_data_2022
historic_demand_data_2023
historic_demand_data_2024
historic_demand_data_2025
historic_demand_data_2001
historic_demand_data_2002
historic_demand_data_2003
historic_demand_data_2004
historic_demand_data_2005
historic_demand_data_2006
historic_demand_data_2007
historic_demand_data_2008
historic_demand_data_2026


In [21]:
import requests
from pathlib import Path

energy_dir = Path("../data/raw/energy")
energy_dir.mkdir(parents=True, exist_ok=True)

for r in resources:

    if "historic_demand_data_" not in r["name"]:
        continue

    year = r["name"].split("_")[-1]

    url = r["path"]

    output_file = energy_dir / f"demanddata_{year}.csv"

    print(f"Downloading {year}...")

    response = requests.get(url)

    with open(output_file, "wb") as f:
        f.write(response.content)

print("\nAll downloads completed.")


All downloads completed.


In [22]:
import os

files = sorted(os.listdir("../data/raw/energy"))

print("Total Files:", len(files))

for file in files:
    print(file)

Total Files: 27
demanddata_2001.csv
demanddata_2002.csv
demanddata_2003.csv
demanddata_2004.csv
demanddata_2005.csv
demanddata_2006.csv
demanddata_2007.csv
demanddata_2008.csv
demanddata_2009.csv
demanddata_2010.csv
demanddata_2011.csv
demanddata_2012.csv
demanddata_2013.csv
demanddata_2014.csv
demanddata_2015.csv
demanddata_2016.csv
demanddata_2017.csv
demanddata_2018.csv
demanddata_2019.csv
demanddata_2020.csv
demanddata_2021.csv
demanddata_2022.csv
demanddata_2023.csv
demanddata_2024.csv
demanddata_2025.csv
demanddata_2026.csv
historic_demand_2026.csv


In [23]:
import os

duplicate_file = "../data/raw/energy/historic_demand_2026.csv"

if os.path.exists(duplicate_file):
    os.remove(duplicate_file)
    print("Duplicate file removed")
else:
    print("File not found")

Duplicate file removed


In [24]:
import os

files = sorted(os.listdir("../data/raw/energy"))

print("Total Files:", len(files))

Total Files: 26


In [25]:
import pandas as pd
import glob

csv_files = glob.glob("../data/raw/energy/*.csv")

dfs = []

for file in csv_files:

    print("Loading:", file)

    temp = pd.read_csv(file)

    temp["source_file"] = file

    dfs.append(temp)

master_df = pd.concat(dfs, ignore_index=True)

print("\nCombined Dataset Shape:")
print(master_df.shape)

Loading: ../data/raw/energy\demanddata_2001.csv
Loading: ../data/raw/energy\demanddata_2002.csv
Loading: ../data/raw/energy\demanddata_2003.csv
Loading: ../data/raw/energy\demanddata_2004.csv
Loading: ../data/raw/energy\demanddata_2005.csv
Loading: ../data/raw/energy\demanddata_2006.csv
Loading: ../data/raw/energy\demanddata_2007.csv
Loading: ../data/raw/energy\demanddata_2008.csv
Loading: ../data/raw/energy\demanddata_2009.csv
Loading: ../data/raw/energy\demanddata_2010.csv
Loading: ../data/raw/energy\demanddata_2011.csv
Loading: ../data/raw/energy\demanddata_2012.csv
Loading: ../data/raw/energy\demanddata_2013.csv
Loading: ../data/raw/energy\demanddata_2014.csv
Loading: ../data/raw/energy\demanddata_2015.csv
Loading: ../data/raw/energy\demanddata_2016.csv
Loading: ../data/raw/energy\demanddata_2017.csv
Loading: ../data/raw/energy\demanddata_2018.csv
Loading: ../data/raw/energy\demanddata_2019.csv
Loading: ../data/raw/energy\demanddata_2020.csv
Loading: ../data/raw/energy\demanddata_2

In [26]:
master_df.head()

,SETTLEMENT_DATE,SETTLEMENT_PERIOD,ND,TSD,ENGLAND_WALES_DEMAND,EMBEDDED_WIND_GENERATION,EMBEDDED_WIND_CAPACITY,EMBEDDED_SOLAR_GENERATION,EMBEDDED_SOLAR_CAPACITY,NON_BM_STOR,...,IFA2_FLOW,BRITNED_FLOW,MOYLE_FLOW,EAST_WEST_FLOW,NEMO_FLOW,NSL_FLOW,ELECLINK_FLOW,VIKING_FLOW,GREENLINK_FLOW,source_file
0,2001-01-01,1,38631,NaN,34060,NaN,NaN,NaN,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,../data/raw/energy\demanddata_2001.csv
1,2001-01-01,2,39808,NaN,35370,NaN,NaN,NaN,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,../data/raw/energy\demanddata_2001.csv
2,2001-01-01,3,40039,NaN,35680,NaN,NaN,NaN,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,../data/raw/energy\demanddata_2001.csv
3,2001-01-01,4,39339,NaN,35029,NaN,NaN,NaN,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,../data/raw/energy\demanddata_2001.csv
4,2001-01-01,5,38295,NaN,34047,NaN,NaN,NaN,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,../data/raw/energy\demanddata_2001.csv


In [27]:
master_df.columns.tolist()

['SETTLEMENT_DATE',
 'SETTLEMENT_PERIOD',
 'ND',
 'TSD',
 'ENGLAND_WALES_DEMAND',
 'EMBEDDED_WIND_GENERATION',
 'EMBEDDED_WIND_CAPACITY',
 'EMBEDDED_SOLAR_GENERATION',
 'EMBEDDED_SOLAR_CAPACITY',
 'NON_BM_STOR',
 'PUMP_STORAGE_PUMPING',
 'SCOTTISH_TRANSFER',
 'IFA_FLOW',
 'IFA2_FLOW',
 'BRITNED_FLOW',
 'MOYLE_FLOW',
 'EAST_WEST_FLOW',
 'NEMO_FLOW',
 'NSL_FLOW',
 'ELECLINK_FLOW',
 'VIKING_FLOW',
 'GREENLINK_FLOW',
 'source_file']

In [28]:
master_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 444670 entries, 0 to 444669
Data columns (total 23 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   SETTLEMENT_DATE            444670 non-null  str    
 1   SETTLEMENT_PERIOD          444670 non-null  int64  
 2   ND                         444670 non-null  int64  
 3   TSD                        374542 non-null  float64
 4   ENGLAND_WALES_DEMAND       444670 non-null  int64  
 5   EMBEDDED_WIND_GENERATION   339502 non-null  float64
 6   EMBEDDED_WIND_CAPACITY     339502 non-null  float64
 7   EMBEDDED_SOLAR_GENERATION  304414 non-null  float64
 8   EMBEDDED_SOLAR_CAPACITY    304414 non-null  float64
 9   NON_BM_STOR                444670 non-null  int64  
 10  PUMP_STORAGE_PUMPING       444670 non-null  int64  
 11  SCOTTISH_TRANSFER          58990 non-null   float64
 12  IFA_FLOW                   444670 non-null  int64  
 13  IFA2_FLOW                  304414 non-nu

In [29]:
master_df.to_csv(
    "../data/processed/uk_energy_master.csv",
    index=False
)

print("Master dataset saved successfully")

Master dataset saved successfully


In [30]:
master_df.shape

(444670, 23)

In [31]:
master_df.columns.tolist()

['SETTLEMENT_DATE',
 'SETTLEMENT_PERIOD',
 'ND',
 'TSD',
 'ENGLAND_WALES_DEMAND',
 'EMBEDDED_WIND_GENERATION',
 'EMBEDDED_WIND_CAPACITY',
 'EMBEDDED_SOLAR_GENERATION',
 'EMBEDDED_SOLAR_CAPACITY',
 'NON_BM_STOR',
 'PUMP_STORAGE_PUMPING',
 'SCOTTISH_TRANSFER',
 'IFA_FLOW',
 'IFA2_FLOW',
 'BRITNED_FLOW',
 'MOYLE_FLOW',
 'EAST_WEST_FLOW',
 'NEMO_FLOW',
 'NSL_FLOW',
 'ELECLINK_FLOW',
 'VIKING_FLOW',
 'GREENLINK_FLOW',
 'source_file']

In [32]:
pip install pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [33]:
import pandas as pd

df = pd.read_csv(
    "../data/processed/uk_energy_master.csv"
)

df.to_parquet(
    "../data/processed/uk_energy_master.parquet",
    engine="pyarrow",
    index=False
)

print("Parquet file created successfully")

Parquet file created successfully


In [34]:
master_df.to_parquet(
    "../data/processed/uk_energy_master.parquet",
    engine="pyarrow",
    index=False
)

print("Parquet file saved successfully")

Parquet file saved successfully


In [35]:
parquet_df = pd.read_parquet(
    "../data/processed/uk_energy_master.parquet"
)

print(parquet_df.shape)


(444670, 23)


In [36]:
import os

csv_size = os.path.getsize(
    "../data/processed/uk_energy_master.csv"
) / (1024**2)

parquet_size = os.path.getsize(
    "../data/processed/uk_energy_master.parquet"
) / (1024**2)

print(f"CSV Size: {csv_size:.2f} MB")
print(f"Parquet Size: {parquet_size:.2f} MB")

CSV Size: 56.91 MB
Parquet Size: 6.22 MB
